In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType


In [0]:
dbutils.widgets.text("silver_catalog", "dbr_dev")
dbutils.widgets.text("silver_schema", "artemzharkov10_silver")

dbutils.widgets.text("gold_catalog", "dbr_dev")
dbutils.widgets.text("gold_schema", "artemzharkov10_gold")


SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

In [0]:
SILVER_ACCIDENTS_TABLE = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_sewik_accidents"
GOLD_WEATHER_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_weather_clusters"

GOLD_OUTPUT_TABLE = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_accident_weather_analytics"

In [0]:
df_silver_accidents = spark.table(SILVER_ACCIDENTS_TABLE)
df_gold_weather_clasters = spark.table(GOLD_WEATHER_TABLE)

In [0]:
# Mathematically round the `accident_timestamp` to the nearest 1 hours (3,600 seconds)
df_accidents_prepared = df_silver_accidents.withColumn(
    "join_time",
    F.from_unixtime(
        #  The accident is attributed to the upcoming temporary weather observation. 
        F.round(F.unix_timestamp(F.col("accident_timestamp")) / 3600) * 3600 # seconds
    ).cast(TimestampType())
)

df_weather_prepared = df_gold_weather_clasters.withColumnRenamed("time", "join_time")

df_gold_merged = df_accidents_prepared.join(
    df_weather_prepared,
    on=["voivodeship", "join_time"],
    how="left"
)

In [0]:
df_gold_final = df_gold_merged.drop(
    "gps_x",
    "gps_y",
    "gps_x_gus",
    "gps_y_gus",
    "report_timestamp",
    "arrival_timestamp",
    "geod_code",
    "built_up_area_code",
    "light_conditions_code",
    "traffic_lights_code",
    "police_force_code",
    "police_unit_local",
    "police_unit_handling",
    "police_unit_operator",
    "public_road",
    "speed_limit",
    "municipality",
    "district",
    "city"
)

df_gold_final = df_gold_final.filter(F.col("weather_claster").isNotNull())

(
    df_gold_final.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("voivodeship")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_OUTPUT_TABLE)
)

In [0]:
# print(df_gold_final.count())
# display(df_gold_final)